# 🛡️ Cyber Threat Intelligence Dashboard
## Network Traffic Anomaly Detection

**Author:** Pramod Prakash Jadhav  
**Course:** AI & ML Essentials — IIT Patna (Vishlesan i-Hub)  
**Portfolio:** [pramodjadhav.vercel.app](https://pramodjadhav.vercel.app)  
**GitHub:** [github.com/pramodj551-oss](https://github.com/pramodj551-oss)

---

### 📌 Project Overview
This notebook demonstrates **real-world network traffic anomaly detection** using:
- **Isolation Forest** — unsupervised ML for anomaly detection
- **Simulated network logs** — 10,000+ packets with realistic attack patterns
- **Interactive Plotly charts** — traffic visualization, geo maps, attack classification
- **IP Reputation Scoring** — rule-based threat scoring system

### 🧠 ML Concepts Used
| Concept | Application |
|---------|-------------|
| Isolation Forest | Unsupervised anomaly detection |
| Feature Engineering | Packet size, port risk, frequency |
| Severity Classification | Rule-based threat scoring |
| Data Visualization | Plotly interactive charts |

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries
!pip install plotly pandas numpy scikit-learn -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import random
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print('✅ All libraries imported successfully!')
print(f'Pandas: {pd.__version__}')
print(f'NumPy:  {np.__version__}')

## 🗄️ Step 2 — Generate Simulated Network Traffic Data

We simulate **10,000 network packets** with realistic features:
- **Normal traffic** (85%) — regular HTTP, HTTPS, DNS traffic
- **Attack traffic** (15%) — SQL Injection, DDoS, Brute Force, Port Scan, Phishing, Malware C2

In [ ]:
# ── Constants ──────────────────────────────────────────────────
ATTACK_TYPES = ['SQL Injection', 'Brute Force', 'DDoS', 'Port Scan', 'Phishing', 'Malware C2']
COUNTRIES    = ['Russia', 'China', 'USA', 'N.Korea', 'Iran', 'Ukraine', 'Nigeria', 'Germany', 'Brazil', 'India']
COUNTRY_COORDS = {
    'Russia':  (55.75, 37.61), 'China':   (39.90, 116.40),
    'USA':     (38.90, -77.03),'N.Korea': (39.02, 125.75),
    'Iran':    (35.68,  51.42),'Ukraine': (50.45,  30.52),
    'Nigeria': ( 9.07,   7.40),'Germany': (52.52,  13.40),
    'Brazil':  (-15.78,-47.92),'India':   (28.61,  77.20),
}
PROTOCOLS    = ['TCP', 'UDP', 'HTTP', 'HTTPS', 'DNS', 'FTP', 'SSH']
RISKY_PORTS  = [22, 23, 3389, 445, 1433, 3306, 8080, 4444]
NORMAL_PORTS = [80, 443, 53, 8443, 8080]

def generate_ip(country):
    """Generate realistic IP based on country risk profile"""
    high_risk = ['Russia','China','N.Korea','Iran','Ukraine','Nigeria']
    if country in high_risk:
        return f"{random.randint(80,220)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}"
    return f"{random.randint(1,79)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}"

def generate_network_data(n_samples=10000):
    """Generate simulated network traffic with attacks embedded"""
    data = []
    base_time = datetime.now() - timedelta(hours=2)

    for i in range(n_samples):
        is_attack = random.random() < 0.15   # 15% attack traffic
        country   = random.choice(COUNTRIES)
        timestamp = base_time + timedelta(seconds=i * 0.72)

        if is_attack:
            attack_type = random.choice(ATTACK_TYPES)
            # Attack packets have distinctive features
            if attack_type == 'DDoS':
                pkt_size    = random.randint(40, 120)      # small packets, high volume
                pkt_count   = random.randint(500, 2000)
                dst_port    = random.choice([80, 443])
                duration    = random.uniform(0.001, 0.01)
            elif attack_type == 'Brute Force':
                pkt_size    = random.randint(100, 300)
                pkt_count   = random.randint(200, 800)
                dst_port    = random.choice([22, 3389, 21])
                duration    = random.uniform(0.5, 2.0)
            elif attack_type == 'Port Scan':
                pkt_size    = random.randint(40, 80)
                pkt_count   = random.randint(100, 500)
                dst_port    = random.randint(1, 65535)
                duration    = random.uniform(0.001, 0.1)
            elif attack_type == 'SQL Injection':
                pkt_size    = random.randint(800, 3000)    # large payload
                pkt_count   = random.randint(5, 50)
                dst_port    = random.choice([80, 443, 8080])
                duration    = random.uniform(0.1, 1.0)
            elif attack_type == 'Malware C2':
                pkt_size    = random.randint(200, 600)
                pkt_count   = random.randint(10, 100)
                dst_port    = random.choice([4444, 8888, 1337, 6667])
                duration    = random.uniform(1.0, 10.0)
            else:  # Phishing
                pkt_size    = random.randint(400, 1500)
                pkt_count   = random.randint(1, 20)
                dst_port    = random.choice([80, 443, 25, 587])
                duration    = random.uniform(0.5, 3.0)

            rep_score = random.randint(1, 30)   # low reputation = suspicious
        else:
            attack_type = 'Normal'
            pkt_size    = random.randint(200, 1500)
            pkt_count   = random.randint(1, 50)
            dst_port    = random.choice(NORMAL_PORTS)
            duration    = random.uniform(0.1, 5.0)
            rep_score   = random.randint(50, 100)

        lat, lon = COUNTRY_COORDS[country]
        data.append({
            'timestamp':    timestamp,
            'src_ip':       generate_ip(country),
            'dst_port':     dst_port,
            'protocol':     random.choice(PROTOCOLS),
            'packet_size':  pkt_size,
            'packet_count': pkt_count,
            'duration':     round(duration, 4),
            'rep_score':    rep_score,
            'country':      country,
            'lat':          lat + random.uniform(-3, 3),
            'lon':          lon + random.uniform(-3, 3),
            'true_label':   attack_type,
            'is_attack':    1 if is_attack else 0,
        })

    return pd.DataFrame(data)

df = generate_network_data(10000)
print(f'✅ Dataset generated: {len(df):,} network packets')
print(f'   Normal traffic : {(df.is_attack==0).sum():,} packets ({(df.is_attack==0).mean()*100:.1f}%)')
print(f'   Attack traffic : {(df.is_attack==1).sum():,} packets ({(df.is_attack==1).mean()*100:.1f}%)')
print()
df.head(10)

## 🔬 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('📊 Dataset Statistics')
print('='*50)
print(f'Total packets      : {len(df):,}')
print(f'Time range         : {df.timestamp.min().strftime("%H:%M:%S")} → {df.timestamp.max().strftime("%H:%M:%S")}')
print(f'Unique source IPs  : {df.src_ip.nunique():,}')
print(f'Unique countries   : {df.country.nunique()}')
print()
print('Attack Type Distribution:')
attack_dist = df[df.is_attack==1]['true_label'].value_counts()
for atk, cnt in attack_dist.items():
    bar = '█' * int(cnt/20)
    print(f'  {atk:<20} {cnt:>4}  {bar}')

print()
print('Traffic by Country (Top 5):')
print(df['country'].value_counts().head())

In [ ]:
# ── Chart 1: Traffic over time ──────────────────────────────
df['minute'] = df['timestamp'].dt.floor('1min')
traffic_time = df.groupby(['minute','is_attack']).size().unstack(fill_value=0).reset_index()
traffic_time.columns = ['minute','normal','attack']

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=traffic_time['minute'], y=traffic_time['normal'],
    name='Normal', fill='tozeroy',
    line=dict(color='#378add', width=2),
    fillcolor='rgba(55,138,221,0.1)'
))
fig1.add_trace(go.Scatter(
    x=traffic_time['minute'], y=traffic_time['attack'],
    name='Malicious', fill='tozeroy',
    line=dict(color='#e24b4a', width=2),
    fillcolor='rgba(226,75,74,0.1)'
))
fig1.update_layout(
    title='📈 Network Traffic — Normal vs Malicious (per minute)',
    xaxis_title='Time', yaxis_title='Packet Count',
    template='plotly_dark', height=350,
    legend=dict(orientation='h', y=1.1)
)
fig1.show()

In [ ]:
# ── Chart 2: Attack type distribution ──────────────────────
atk_counts = df[df.is_attack==1]['true_label'].value_counts().reset_index()
atk_counts.columns = ['Attack Type', 'Count']
colors = ['#e24b4a','#f59e0b','#534ab7','#378add','#993556','#1d9e75']

fig2 = go.Figure(go.Bar(
    y=atk_counts['Attack Type'],
    x=atk_counts['Count'],
    orientation='h',
    marker_color=colors,
    text=atk_counts['Count'],
    textposition='outside'
))
fig2.update_layout(
    title='📊 Attack Type Classifier — Detected Threats',
    xaxis_title='Count', yaxis_title='',
    template='plotly_dark', height=350
)
fig2.show()

In [ ]:
# ── Chart 3: Attack origin world map ───────────────────────
attack_df = df[df.is_attack==1].copy()

fig3 = px.scatter_geo(
    attack_df.sample(min(500, len(attack_df))),
    lat='lat', lon='lon',
    color='true_label',
    hover_name='country',
    hover_data={'src_ip':True, 'true_label':True, 'lat':False, 'lon':False},
    color_discrete_map={
        'SQL Injection':'#e24b4a', 'Brute Force':'#f59e0b',
        'DDoS':'#534ab7',          'Port Scan':'#378add',
        'Phishing':'#993556',      'Malware C2':'#1d9e75'
    },
    title='🌍 Attack Origin Map — Geo Distribution of Threats',
    projection='natural earth'
)
fig3.update_layout(
    template='plotly_dark', height=420,
    geo=dict(
        showland=True,  landcolor='rgba(0,229,160,0.08)',
        showocean=True, oceancolor='rgba(8,12,16,0.9)',
        showcoastlines=True, coastlinecolor='rgba(0,229,160,0.3)',
        showcountries=True, countrycolor='rgba(100,116,139,0.3)',
    )
)
fig3.show()

In [ ]:
# ── Chart 4: Packet size distribution ──────────────────────
fig4 = go.Figure()
fig4.add_trace(go.Histogram(
    x=df[df.is_attack==0]['packet_size'],
    name='Normal', nbinsx=50,
    marker_color='rgba(55,138,221,0.7)',
    opacity=0.75
))
fig4.add_trace(go.Histogram(
    x=df[df.is_attack==1]['packet_size'],
    name='Malicious', nbinsx=50,
    marker_color='rgba(226,75,74,0.7)',
    opacity=0.75
))
fig4.update_layout(
    title='📦 Packet Size Distribution — Normal vs Malicious',
    xaxis_title='Packet Size (bytes)',
    yaxis_title='Frequency',
    barmode='overlay',
    template='plotly_dark', height=350
)
fig4.show()

## 🤖 Step 4 — ML Model: Isolation Forest for Anomaly Detection

**Why Isolation Forest?**
- Unsupervised — no labelled data needed (perfect for real SOC scenarios)
- Works by isolating anomalies rather than profiling normal behaviour
- Fast and effective on high-dimensional network data
- Used in production SOC environments

In [ ]:
# ── Feature Engineering ────────────────────────────────────
def is_risky_port(port):
    return 1 if port in [22,23,3389,445,1433,3306,4444,8888,1337,6667] else 0

def get_bytes_per_sec(row):
    return (row['packet_size'] * row['packet_count']) / max(row['duration'], 0.001)

df['risky_port']    = df['dst_port'].apply(is_risky_port)
df['bytes_per_sec'] = df.apply(get_bytes_per_sec, axis=1)
df['log_pkt_count'] = np.log1p(df['packet_count'])
df['log_pkt_size']  = np.log1p(df['packet_size'])
df['low_rep']       = (df['rep_score'] < 30).astype(int)

# Features used for ML
FEATURES = ['packet_size','packet_count','duration','rep_score',
            'risky_port','bytes_per_sec','log_pkt_count','log_pkt_size','low_rep']

X = df[FEATURES].values
y = df['is_attack'].values

# Scale features
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('✅ Features engineered:', FEATURES)
print(f'   Dataset shape: {X_scaled.shape}')

In [ ]:
# ── Train Isolation Forest ─────────────────────────────────
print('🤖 Training Isolation Forest...')

model = IsolationForest(
    n_estimators=200,
    contamination=0.15,   # Expected 15% anomalies
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)
model.fit(X_scaled)

# Predict: -1 = anomaly (attack), 1 = normal
preds_raw   = model.predict(X_scaled)
df['predicted_anomaly'] = (preds_raw == -1).astype(int)
df['anomaly_score']     = -model.score_samples(X_scaled)   # higher = more anomalous

print('✅ Model trained!')
print(f'   Anomalies detected : {df.predicted_anomaly.sum():,}')
print(f'   Normal classified  : {(df.predicted_anomaly==0).sum():,}')

In [ ]:
# ── Model Evaluation ──────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

acc  = accuracy_score(y, df['predicted_anomaly'])
prec = precision_score(y, df['predicted_anomaly'])
rec  = recall_score(y, df['predicted_anomaly'])
f1   = f1_score(y, df['predicted_anomaly'])

print('📊 Model Performance')
print('='*40)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  Precision : {prec:.4f}  ({prec*100:.1f}%)')
print(f'  Recall    : {rec:.4f}  ({rec*100:.1f}%)')
print(f'  F1 Score  : {f1:.4f}  ({f1*100:.1f}%)')
print()
print('Classification Report:')
print(classification_report(y, df['predicted_anomaly'],
                             target_names=['Normal','Attack']))

In [ ]:
# ── Chart 5: Anomaly Score Distribution ────────────────────
fig5 = go.Figure()
fig5.add_trace(go.Histogram(
    x=df[df.is_attack==0]['anomaly_score'],
    name='Normal',   nbinsx=60,
    marker_color='rgba(55,138,221,0.7)', opacity=0.75
))
fig5.add_trace(go.Histogram(
    x=df[df.is_attack==1]['anomaly_score'],
    name='Attack',   nbinsx=60,
    marker_color='rgba(226,75,74,0.7)',  opacity=0.75
))
fig5.update_layout(
    title='🎯 Isolation Forest — Anomaly Score Distribution',
    xaxis_title='Anomaly Score (higher = more suspicious)',
    yaxis_title='Count', barmode='overlay',
    template='plotly_dark', height=350
)
fig5.show()

In [ ]:
# ── Chart 6: Confusion Matrix ──────────────────────────────
cm = confusion_matrix(y, df['predicted_anomaly'])
fig6 = go.Figure(go.Heatmap(
    z=cm, x=['Predicted Normal','Predicted Attack'],
    y=['Actual Normal','Actual Attack'],
    colorscale='Teal', showscale=True,
    text=cm, texttemplate='%{text}', textfont=dict(size=16)
))
fig6.update_layout(
    title='🎯 Confusion Matrix — Isolation Forest',
    template='plotly_dark', height=350
)
fig6.show()

## 🚨 Step 5 — Threat Severity Classification

In [ ]:
# Assign severity based on anomaly score + features
def assign_severity(row):
    score = row['anomaly_score']
    if row['predicted_anomaly'] == 0:
        return 'NORMAL'
    if score > 0.65 or row['rep_score'] < 10:
        return 'CRITICAL'
    elif score > 0.50 or row['risky_port'] == 1:
        return 'HIGH'
    elif score > 0.38:
        return 'MEDIUM'
    else:
        return 'LOW'

df['severity'] = df.apply(assign_severity, axis=1)

sev_counts = df[df.severity!='NORMAL']['severity'].value_counts()
print('🚨 Threat Severity Summary')
print('='*35)
for sev, cnt in sev_counts.items():
    bar = '█' * int(cnt / 15)
    print(f'  {sev:<10} {cnt:>5}  {bar}')

# Severity pie chart
fig7 = go.Figure(go.Pie(
    labels=sev_counts.index,
    values=sev_counts.values,
    hole=0.45,
    marker_colors=['#e24b4a','#f59e0b','#378add','#00e5a0'],
    textinfo='label+percent'
))
fig7.update_layout(
    title='🚨 Threat Severity Distribution',
    template='plotly_dark', height=380
)
fig7.show()

## 🔎 Step 6 — IP Reputation Scoring

In [ ]:
# Top malicious IPs by anomaly score
top_threats = (
    df[df.predicted_anomaly==1]
    .groupby('src_ip')
    .agg(
        total_packets  = ('src_ip','count'),
        avg_score      = ('anomaly_score','mean'),
        avg_rep        = ('rep_score','mean'),
        country        = ('country','first'),
        attack_types   = ('true_label', lambda x: ', '.join(x.unique()[:2]))
    )
    .sort_values('avg_score', ascending=False)
    .head(10)
    .reset_index()
)

top_threats['avg_score'] = top_threats['avg_score'].round(4)
top_threats['avg_rep']   = top_threats['avg_rep'].round(1)
top_threats['status']    = top_threats['avg_rep'].apply(
    lambda s: '🔴 Malicious' if s<25 else '⚠️ Suspicious' if s<60 else '✅ Clean'
)

print('🔎 Top 10 Threat IPs — IP Reputation Report')
print('='*70)
display(top_threats[['src_ip','country','total_packets','avg_score','avg_rep','status','attack_types']])

## 📋 Step 7 — Final SOC Summary Report

In [ ]:
total       = len(df)
detected    = df['predicted_anomaly'].sum()
critical    = (df['severity']=='CRITICAL').sum()
high_sev    = (df['severity']=='HIGH').sum()
health      = max(45, 100 - int((critical/total)*150))
top_country = df[df.predicted_anomaly==1]['country'].value_counts().idxmax()
top_attack  = df[df.predicted_anomaly==1]['true_label'].value_counts().idxmax()

print('=' * 55)
print('  🛡️  CYBER THREAT INTELLIGENCE — SOC REPORT')
print('=' * 55)
print(f'  Analyst        : Pramod Prakash Jadhav')
print(f'  Course         : AI & ML Essentials — IIT Patna')
print(f'  Generated      : {datetime.now().strftime("%d %b %Y %H:%M:%S")}')
print('-' * 55)
print(f'  Total packets  : {total:,}')
print(f'  Threats found  : {detected:,}  ({detected/total*100:.1f}%)')
print(f'  CRITICAL       : {critical:,}')
print(f'  HIGH           : {high_sev:,}')
print(f'  Network health : {health}%')
print(f'  Top origin     : {top_country}')
print(f'  Top attack     : {top_attack}')
print(f'  Model          : Isolation Forest (n=200, contam=15%)')
print(f'  Precision      : {prec*100:.1f}%')
print(f'  Recall         : {rec*100:.1f}%')
print(f'  F1 Score       : {f1*100:.1f}%')
print('=' * 55)
print('  ✅ Analysis complete — Stay Secure!')
print('=' * 55)

---
## 🔗 Project Links

| Resource | Link |
|----------|------|
| 🌐 Portfolio | [pramodjadhav.vercel.app](https://pramodjadhav.vercel.app) |
| 🐙 GitHub | [github.com/pramodj551-oss](https://github.com/pramodj551-oss) |
| 💼 LinkedIn | [Pramod Prakash Jadhav](https://www.linkedin.com/in/pramod-prakash-jadhav-42ba2281) |

**Course:** AI & ML Essentials — IIT Patna (Vishlesan i-Hub) · 2025–2026